In [1]:
import os
import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd

from shapely.geometry import LineString, MultiLineString
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
from shapely.geometry import box
import matplotlib.colors as mcolors

from dask import delayed, compute
from tqdm import tqdm
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar

# ============================
# User settings
# ============================

# Generator site list
gen_csv = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_info.csv")

# BARRA-C2 variable paths
u_path = "/g/data/ob53/BARRA2/output/reanalysis/AUST-04/BOM/ERA5/historical/hres/BARRA-C2/v1/1hr/ua100m/latest/"
v_path = "/g/data/ob53/BARRA2/output/reanalysis/AUST-04/BOM/ERA5/historical/hres/BARRA-C2/v1/1hr/va100m/latest/"


In [2]:
client = Client(n_workers=18,
    threads_per_worker=1,
    memory_limit=f"{int(7)}GB"
)

client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 18
Total threads: 18,Total memory: 117.35 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:41253,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:44487,Total threads: 1
Dashboard: /proxy/34829/status,Memory: 6.52 GiB
Nanny: tcp://127.0.0.1:37827,


2025-10-13 12:32:19,762 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle a410493855d4cac2b9291701091b6ec5 initialized by task ('rechunk-merge-rechunk-transfer-ce519526a24c2de64c1e4b76307d51ea', 0, 0, 3, 20, 0, 12) executed on worker tcp://127.0.0.1:33205
2025-10-13 12:32:21,314 - distributed.nanny.memory - WARNING - Worker tcp://127.0.0.1:33205 (pid=26227) exceeded 95% memory budget. Restarting...
2025-10-13 12:32:21,433 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:33205' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('rechunk-split-19a7203f889799c8e36a5e0ccd9384e3', 80349), ('rechunk-split-19a7203f889799c8e36a5e0ccd9384e3', 80346), ('rechunk-split-19a7203f889799c8e36a5e0ccd9384e3', 80352), ('rechunk-split-19a7203f889799c8e36a5e0ccd9384e3', 80358), ('rechunk-split-19a7203f889799c8e36a5e0ccd9384e3', 80355), ('rechunk-split-19a7203f889799c8e36a5e0ccd9384e3', 80361), ('rechunk-split-19a7203f889799c8e36a5e0ccd93

In [3]:
sample = 'baseline'
# reanalysis = 'BARRA-C2'

if sample == 'heatwave':
    cluster_dates = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/alpine_cluster.csv")
    output_dir = "/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/hw"
    mode_str = 'heatwave'
elif sample == 'baseline':
    cluster_dates = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/no_hw_alpine_cluster.csv")
    output_dir = "/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/no_hw"
    mode_str = 'baseline'

In [4]:
# Statistically significant w min sample size = 20, and Mann-Whitney 
cluster = ['TARALGA1','CROOKWF2',
            'GULLRWF1',
            'GUNNING1',
            'CRURWF1',
            'WOODLWN1',
            'BOCORWF1',
            'BODWF1']

cluster = gen_csv[gen_csv['DUID'].isin(cluster)][['DUID','lat','lon']]

extent = [147, 152, -38.5, -32]
lon_min, lon_max, lat_min, lat_max = extent

In [5]:
def get_days(days, nc_dir):
    date_list = pd.to_datetime(days['date'])

    # Build filename filter
    all_files = os.listdir(nc_dir)
    selected_files = [
        os.path.join(nc_dir, f)
        for f in all_files
        if any(d.strftime("%Y%m") in f for d in date_list)
    ]

    # Open multiple files lazily with parallel reads
    ds = xr.open_mfdataset(
        selected_files,
        combine='by_coords',
        parallel=True,
        chunks='auto'
    )

    # Select all hours of the requested dates
    subset = ds.where(ds.time.dt.floor('D').isin(date_list), drop=True)

    return subset

In [6]:
def get_ds():
    u_hw_cluster = get_days(cluster_dates, u_path)
    v_hw_cluster = get_days(cluster_dates, v_path)
    
    ds = xr.merge([u_hw_cluster, v_hw_cluster])
    ds = ds.chunk({'time': -1,'lat': 80,'lon':50})
    
    # Convert to Australia/Sydney
    local_time = (
        pd.DatetimeIndex(ds.time.values)
        .tz_localize("UTC")
        .tz_convert("Australia/Sydney")
    )
    
    # Drop tzinfo so xarray can store it
    local_time_naive = local_time.tz_localize(None)
    
    # Assign back to dataset
    ds = ds.assign_coords(time=local_time_naive)
    return ds


In [7]:
ds = get_ds()
# Crop to extent
ds_subset = ds.sel(**{
    'lon': slice(lon_min, lon_max),
    'lat': slice(lat_min, lat_max)
})
ds_subset = ds_subset.chunk(chunks='auto')

In [8]:
# This computes the composite of variance
ds_subset['windspeed'] = np.sqrt(ds_subset['ua100m']**2 + ds_subset['va100m']**2)

def temporal_variance(x):
    # x has dimensions (time, lat, lon)
    return x.var(dim="time", ddof=1)  # use ddof=1 for unbiased sample variance

hourly_var_per_point = ds_subset['windspeed'].groupby("time.hour").map(temporal_variance)
hourly_var_per_point = hourly_var_per_point.compute()

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.08/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 189.38 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
2025-10-13 12:32:20,741 - distributed.worker.memory - WARNING - Worker is at 82% memory usage. Pausing worker.  Process memory: 5.40 GiB -- Worker memory limit: 6.52 GiB
2025-10-13 12:32:20,962 - distributed.worker.memory - WARNING - Worker is at 80% memory usage. Pausing worker.  Process memory: 5.27 GiB -- Worker memory limit: 6.52 GiB
2025-10-13 12:32:21,779 - distributed.worker - ERROR - Worker stream died during communication: tcp://127.0.0.1:38247
Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis

KilledWorker: Attempted to run task ('rechunk-merge-rechunk-transfer-ce519526a24c2de64c1e4b76307d51ea', 0, 0, 3, 20, 0, 12) on 4 different workers, but all those workers died while running it. The last worker that attempt to run the task was tcp://127.0.0.1:46097. Inspecting worker logs is often a good next step to diagnose what went wrong. For more information see https://distributed.dask.org/en/stable/killed.html.

In [ ]:
# This computes the hourly u v field composite.
# Lazy hourly mean
hourly_composite =  ds_subset.groupby("time.hour").mean()

# Compute in parallel
with ProgressBar():
    hourly_composite = hourly_composite.persist()
    hourly_composite = hourly_composite.assign_coords(hour=("hour", np.arange(24)))


In [ ]:
# This computes the composites of minimum, maximum, and diunal amplitude
def compute_windspeed_composites(windspeed):
    """
    Compute windspeed composites (max, min, amplitude) from u and v.
    
    Parameters
    ----------
    u, v : xarray.DataArray
        Wind vector components (time x lat x lon)
    
    Returns
    -------
    dict
        {"max": DataArray, "min": DataArray, "diff": DataArray}
    """
    # Compute windspeed
    
    # Compute composites along time dimension
    composite_max = windspeed.max(dim="time").compute()
    composite_min = windspeed.min(dim="time").compute()
    diurnal_amp = (composite_max - composite_min).compute()
    
    return composite_max, composite_min, diurnal_amp

max_speed, min_speed, diurnal_amp = compute_windspeed_composites(ds_subset['windspeed'])

Functions which calculate quantities from vector fields

In [ ]:
def calc_divergence(ds):
    """
    Compute divergence of u/v on a spherical Earth using xarray.
    ds should be a Dataset with dimensions lat x lon (or time x lat x lon).
    """
    R = 6371000  # Earth radius in meters
    
    # lat_rad shape (lat,)
    lat_rad = np.deg2rad(ds['lat'].values)
    
    dx_1d = np.gradient(ds['lon'].values) * (np.pi/180) * R
    dx2d = dx_1d[None, :] * np.cos(lat_rad[:, None])  # shape (lat, lon)
    
    dy2d = np.gradient(ds['lat'].values) * (np.pi/180) * R
    dy2d = dy2d[:, None]  # shape (lat, 1) to broadcast with u/v

    # Compute divergence
    divergence = (ds['ua100m'].differentiate('lon') / dx2d +
                  ds['va100m'].differentiate('lat') / dy2d)
    
    return divergence

In [ ]:
def okubo_weiss(ds, u_name="ua100m", v_name="va100m", lat_name="lat", lon_name="lon"):
    R = 6371000.0
    lat_vals = ds[lat_name].values
    lon_vals = ds[lon_name].values
    lat_rad = np.deg2rad(lat_vals)

    dlat = np.gradient(lat_vals) * np.pi/180 * R
    dlon = np.gradient(lon_vals) * np.pi/180 * R

    # dx/dy 2D arrays matching full grid
    dx2d = xr.DataArray(dlon[None, :] * np.cos(lat_rad[:, None]),
                        dims=[lat_name, lon_name],
                        coords={lat_name: lat_vals, lon_name: lon_vals})
    dy2d = xr.DataArray(dlat[:, None] * np.ones(len(lon_vals)),
                        dims=[lat_name, lon_name],
                        coords={lat_name: lat_vals, lon_name: lon_vals})

    du_dlon = ds[u_name].differentiate(lon_name) * np.pi/180
    du_dlat = ds[u_name].differentiate(lat_name) * np.pi/180
    dv_dlon = ds[v_name].differentiate(lon_name) * np.pi/180
    dv_dlat = ds[v_name].differentiate(lat_name) * np.pi/180

    dudx = du_dlon / dx2d
    dudy = du_dlat / dy2d
    dvdx = dv_dlon / dx2d
    dvdy = dv_dlat / dy2d

    s_n = dudx - dvdy
    s_s = dudy + dvdx
    omega = dvdx - dudy
    OW = s_n**2 + s_s**2 - omega**2

    return xr.Dataset({"s_n": s_n, "s_s": s_s, "omega": omega, "OW": OW})


In [ ]:
def calc_curl(hourly_composite):
    R = 6371000  # Earth radius in meters
        
    # lat_rad shape (lat,)
    lat_rad = np.deg2rad(hourly_composite['lat'].values)
    
    # dx along longitude, in meters
    dx_1d = np.gradient(hourly_composite['lon'].values) * (np.pi/180) * R
    dx2d = dx_1d[None, :] * np.cos(lat_rad[:, None])  # shape (lat, lon)
    
    # dy along latitude, in meters
    dy2d = np.gradient(hourly_composite['lat'].values) * (np.pi/180) * R
    dy2d = dy2d[:, None]  # shape (lat, 1) to broadcast
    
    # Compute curl (z-component)
    curl_z = (hourly_composite['va100m'].differentiate('lon') / dx2d -
              hourly_composite['ua100m'].differentiate('lat') / dy2d)
    return curl_z

Plotting funcitons for vector field, divergence, Okubo-Weiss, and curl

In [ ]:
def plot_variance_frame(field, lat, lon,
                         cluster=None,
                         highlight_id=None,
                         shapefile_gdf=None,
                         output_dir=f"{output_dir}/variance/",
                         extent=extent,
                         vmax=30,
                         t=0,
                         mode_str=mode_str):

    
    # Create figure
    fig, ax = plt.subplots(figsize=(10, 8),
                           subplot_kw={'projection': ccrs.PlateCarree()})
    ax.set_extent(extent, crs=ccrs.PlateCarree())

    # Base map
    ax.add_feature(cfeature.COASTLINE, lw=1.0)
    ax.add_feature(cfeature.STATES, linestyle='-', lw=1.0)
    
    
    # Filled contours
    cf = ax.contourf(lon, lat, field,
                     cmap='viridis', levels=21, vmin=0,vmax=vmax,
                     transform=ccrs.PlateCarree(), zorder=1)
    plt.colorbar(cf, ax=ax, label=f"Windspeed variance [m/s]")
    
    # --- Shapefile contours ---
    if shapefile_gdf is not None and not shapefile_gdf.empty:
        for geom in shapefile_gdf.geometry:
            ax.add_geometries([geom], crs=ccrs.PlateCarree(),
                              facecolor="none", edgecolor="black",
                              linewidth=0.5, zorder=3)
    
    # Cluster points
    if cluster is not None:
        ax.scatter(cluster['lon'], cluster['lat'], color="orange", edgecolor="black", s=50,
                   label="Wind Farms", zorder=4, transform=ccrs.PlateCarree())
        if highlight_id is not None and highlight_id in cluster['DUID'].values:
            highlight_point = cluster.loc[cluster['DUID'] == highlight_id]
            ax.scatter(highlight_point['lon'], highlight_point['lat'],
                       color="red", edgecolor="black", s=60,
                       label=f"ID {highlight_id}", zorder=5, transform=ccrs.PlateCarree())
    
    plt.title(f"Variance of {mode_str} day composite at hour {t}")
    
    # Save
    os.makedirs(output_dir, exist_ok=True)
    filename = os.path.join(output_dir, f"windspeed_var_composite_{t}.png")
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return filename


In [ ]:
def plot_windspeed_frame(field, lat, lon,
                         field_type="max",  # "max", "min", or "diff"
                         cluster=None,
                         highlight_id=None,
                         shapefile_gdf=None,
                         output_dir=f"{output_dir}/minmax/",
                         extent=extent,
                         mode_str=mode_str):

    
    # Create figure
    fig, ax = plt.subplots(figsize=(10, 8),
                           subplot_kw={'projection': ccrs.PlateCarree()})
    ax.set_extent(extent, crs=ccrs.PlateCarree())

    # Base map
    ax.add_feature(cfeature.COASTLINE, lw=1.0)
    ax.add_feature(cfeature.STATES, linestyle='-', lw=1.0)
    
    
    # Filled contours
    cf = ax.contourf(lon, lat, field,
                     cmap='viridis', levels=21,
                     transform=ccrs.PlateCarree(), zorder=1)
    plt.colorbar(cf, ax=ax, label=f"Windspeed ({field_type}) [m/s]")
    
    # --- Shapefile contours ---
    if shapefile_gdf is not None and not shapefile_gdf.empty:
        for geom in shapefile_gdf.geometry:
            ax.add_geometries([geom], crs=ccrs.PlateCarree(),
                              facecolor="none", edgecolor="black",
                              linewidth=0.5, zorder=3)
    
    # Cluster points
    if cluster is not None:
        ax.scatter(cluster['lon'], cluster['lat'], color="orange", edgecolor="black", s=50,
                   label="Wind Farms", zorder=4, transform=ccrs.PlateCarree())
        if highlight_id is not None and highlight_id in cluster['DUID'].values:
            highlight_point = cluster.loc[cluster['DUID'] == highlight_id]
            ax.scatter(highlight_point['lon'], highlight_point['lat'],
                       color="red", edgecolor="black", s=60,
                       label=f"ID {highlight_id}", zorder=5, transform=ccrs.PlateCarree())
    
    # Title
    titles = {
        "max": "Daily Maximum Windspeed",
        "min": "Daily Minimum Windspeed",
        "diff": "Diurnal Amplitude (max-min)",
    }
    plt.title(f"Composite of {titles.get(field_type, field_type)} on {mode_str} days")
    
    # Save
    filename = make_filename(t, output_dir)
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved: {filename}')
    return filename


In [ ]:
def plot_vector_frame(u, v, lat, lon, t, output_dir=f"{output_dir}/C2_hourly_composites/",
               quiver_scale=None, extent=extent, cluster=cluster, highlight_id='BOCORWF1',
                     mode_str=mode_str):
    """
    Plot wind vectors with optional cluster points.
    
    cluster: DataFrame with columns ['ID', 'lat', 'lon']
    highlight_id: specific ID to highlight in red
    """
    speed = np.sqrt(u**2 + v**2)
    plt.figure(figsize=(10,8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, lw=1.5)
    ax.add_feature(cfeature.BORDERS, linestyle='-', lw=1.5)
    ax.add_feature(cfeature.STATES, linestyle='-', lw=1.5)
    
    # Contour of wind speed
    plt.contourf(lon, lat, speed, cmap="cividis", levels=21,
                 vmin=0, vmax=12, transform=ccrs.PlateCarree())
    plt.colorbar(label='Wind speed (m/s)')
    
    # Quiver vectors
    plt.quiver(lon[::step], lat[::step], u[::step, ::step], v[::step, ::step],
               scale=quiver_scale, color='white', transform=ccrs.PlateCarree())
    
    # Plot cluster points
    if cluster is not None:
        # All points in orange
        ax.scatter(cluster['lon'], cluster['lat'], color="orange", alpha=0.6, s=50, label="Wind Farms", zorder=5, transform=ccrs.PlateCarree())
        
        # Highlight one point in red
        if highlight_id is not None and highlight_id in cluster['DUID'].values:
            highlight_point = cluster.loc[cluster['DUID'] == highlight_id]
            ax.scatter(highlight_point['lon'], highlight_point['lat'], 
                       color="red", alpha=0.6, s=80, label=f"ID {highlight_id}", zorder=6,
                       transform=ccrs.PlateCarree())

    plt.title(f'Composite of {mode_str}-day wind vectors at hour {str(t)} (AEST)')
    
    filename = make_filename(t, output_dir)
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved: {filename}')
    return filename


def make_filename(t, output_dir, prefix="wind"):
    try:
        # If t is datetime-like, use date formatting
        dt_str = np.datetime_as_string(t, unit='m')
        dt_str = dt_str.replace('-', '')[2:8] + '_' + dt_str[11:13] + dt_str[14:16]
    except Exception:
        # If it's just an index/hour, format as hour
        if isinstance(t, (int, np.integer)):
            dt_str = f"hour{t:02d}"
        else:
            dt_str = str(t).replace(":", "").replace(" ", "_")
    
    return os.path.join(output_dir, f"{prefix}_{dt_str}.png")


In [ ]:
def plot_divergence_frame(divergence, lat, lon, t, cluster=cluster, highlight_id='BOCORWF1',
                          shapefile_gdf=None,
                          output_dir=f"{output_dir}/divergence_composite",
                          extent=extent,
                          mode_str=mode_str):
    """
    Plot scalar divergence with optional cluster points and contour shapefile.
    """
    fig, ax = plt.subplots(figsize=(10,8), subplot_kw={'projection': ccrs.PlateCarree()})
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    
    # Base map
    ax.add_feature(cfeature.COASTLINE, lw=1.0)
    ax.add_feature(cfeature.STATES, linestyle='-', lw=1.0)
    
    # Divergence field
    # Symmetric normalization around zero
    norm = mcolors.TwoSlopeNorm(vmin=np.nanmin(divergence),
                                vcenter=0,
                                vmax=np.nanmax(divergence))
    
    cf = ax.contourf(lon, lat, divergence, cmap='coolwarm', levels=21, norm=norm,
                     transform=ccrs.PlateCarree(), zorder=1)
    plt.colorbar(cf, ax=ax, label='Divergence (1/s)')
    
    # --- Contours (from shapefile) ---
    if shapefile_gdf is not None and not shapefile_gdf.empty:
        for geom in shapefile_gdf.geometry:
            ax.add_geometries([geom], crs=ccrs.PlateCarree(),
                              facecolor='none', edgecolor='black',
                              linewidth=0.5, zorder=3)
    
    # Cluster points
    if cluster is not None:
        ax.scatter(cluster['lon'], cluster['lat'], color="orange", edgecolor='black', s=50,
                   label="Wind Farms", zorder=4, transform=ccrs.PlateCarree())
        if highlight_id is not None and highlight_id in cluster['DUID'].values:
            highlight_point = cluster.loc[cluster['DUID'] == highlight_id]
            ax.scatter(highlight_point['lon'], highlight_point['lat'],
                       color="red", edgecolor='black', s=60,
                       label=f"ID {highlight_id}", zorder=5, transform=ccrs.PlateCarree())
    
    plt.title(f'Divergence of {mode_str} composite wind field at hour {str(t)} (AEST)')
    
    # Save
    filename = make_filename(t, output_dir)
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return filename

In [ ]:
def plot_ow_frame(df, lat, lon, t, cluster=cluster, highlight_id='BOCORWF1',
                          shapefile_gdf=None,
                          output_dir=f"{output_dir}/okubo-weiss",
                          extent=extent,
                          mode_str=mode_str):
    """
    Plot scalar OW parameter with optional cluster points and contour shapefile.
    """
    fig, ax = plt.subplots(figsize=(10,8), subplot_kw={'projection': ccrs.PlateCarree()})
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    
    # Base map
    ax.add_feature(cfeature.COASTLINE, lw=1.0)
    ax.add_feature(cfeature.STATES, linestyle='-', lw=1.0)
    
    norm = mcolors.TwoSlopeNorm(vmin=np.nanmin(df),
                                vcenter=0,
                                vmax=np.nanmax(df))
    
    cf = ax.contourf(lon, lat, df, cmap='seismic', levels=21, norm=norm,
                     transform=ccrs.PlateCarree(), zorder=1)
    plt.colorbar(cf, ax=ax, label='Okubo-Weiss parameter')
    
    # --- Contours (from shapefile) ---
    if shapefile_gdf is not None and not shapefile_gdf.empty:
        for geom in shapefile_gdf.geometry:
            ax.add_geometries([geom], crs=ccrs.PlateCarree(),
                              facecolor='none', edgecolor='black',
                              linewidth=0.5, zorder=3)
    
    # Cluster points
    if cluster is not None:
        ax.scatter(cluster['lon'], cluster['lat'], color="orange", edgecolor='black', s=50,
                   label="Wind Farms", zorder=4, transform=ccrs.PlateCarree())
        if highlight_id is not None and highlight_id in cluster['DUID'].values:
            highlight_point = cluster.loc[cluster['DUID'] == highlight_id]
            ax.scatter(highlight_point['lon'], highlight_point['lat'],
                       color="red", edgecolor='black', s=60,
                       label=f"ID {highlight_id}", zorder=5, transform=ccrs.PlateCarree())
    
    plt.title(f'Okubo-Weiss on {mode_str} days at hour {str(t)} (AEST)')
    
    # Save
    filename = make_filename(t, output_dir)
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return filename

In [ ]:
def plot_curl_frame(df, lat, lon, t, cluster=cluster, highlight_id='BOCORWF1',
                          shapefile_gdf=None,
                          output_dir=f"{output_dir}/curl",
                          extent=extent,
                          mode_str=mode_str):
    """
    Plot scalar curl parameter with optional cluster points and contour shapefile.
    """
    fig, ax = plt.subplots(figsize=(10,8), subplot_kw={'projection': ccrs.PlateCarree()})
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    
    # Base map
    ax.add_feature(cfeature.COASTLINE, lw=1.0)
    ax.add_feature(cfeature.STATES, linestyle='-', lw=1.0)
    
    norm = mcolors.TwoSlopeNorm(vmin=np.nanmin(df),
                                vcenter=0,
                                vmax=np.nanmax(df))
    
    cf = ax.contourf(lon, lat, df, cmap='PiYG', levels=21, norm=norm,
                     transform=ccrs.PlateCarree(), zorder=1)
    plt.colorbar(cf, ax=ax, label=' Curl (1/s): Pink=CW, Green=ACW ')
    
    # --- Contours (from shapefile) ---
    if shapefile_gdf is not None and not shapefile_gdf.empty:
        for geom in shapefile_gdf.geometry:
            ax.add_geometries([geom], crs=ccrs.PlateCarree(),
                              facecolor='none', edgecolor='black',
                              linewidth=0.5, zorder=3)
    
    # Cluster points
    if cluster is not None:
        ax.scatter(cluster['lon'], cluster['lat'], color="orange", edgecolor='black', s=50,
                   label="Wind Farms", zorder=4, transform=ccrs.PlateCarree())
        if highlight_id is not None and highlight_id in cluster['DUID'].values:
            highlight_point = cluster.loc[cluster['DUID'] == highlight_id]
            ax.scatter(highlight_point['lon'], highlight_point['lat'],
                       color="red", edgecolor='black', s=60,
                       label=f"ID {highlight_id}", zorder=5, transform=ccrs.PlateCarree())
    
    plt.title(f'Curl from {mode_str} day composite at hour {str(t)} (AEST)')
    
    # Save
    filename = make_filename(t, output_dir)
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return filename

Getting shapefile and subset for plotting

In [ ]:
# Load shapefile
gdf = gpd.read_file('/g/data/ng72/ms5578/ID_HW_BARRA/data/raw/contours/aus25cgd_l.shp').to_crs(epsg=4326)
bbox = box(lon_min, lat_min, lon_max, lat_max)
gdf_clip = gdf[gdf.geometry.intersects(bbox)]

In [ ]:
lat_subset = ds_subset['lat'].where(
    (ds_subset['lat'] >= lat_min) & (ds_subset['lat'] <= lat_max),
    drop=True
    ).values

lon_subset = ds_subset['lon'].where(
    (ds_subset['lon'] >= lon_min) & (ds_subset['lon'] <= lon_max),
    drop=True
    ).values

Plotting loops

In [ ]:
for t in range(len(hourly_var_per_point['hour'])):
    time_slice = hourly_var_per_point.isel(hour=t).values
    
    filestring = plot_variance_frame(time_slice, lat_subset, lon_subset, cluster=cluster,
                     shapefile_gdf=gdf, extent=extent, vmax=34.5, t=t)
    
    print(f'Saved variance plots: {filestring}')

In [ ]:
for t in range(len(hourly_composite.hour)):
    step=2
    
    # --- Full resolution (for contours) ---
    u_full = hourly_composite['ua100m'].isel(hour=t).sel(
        lat=lat_subset, lon=lon_subset
    ).values
    v_full = hourly_composite['va100m'].isel(hour=t).sel(
        lat=lat_subset, lon=lon_subset
    ).values

    target_nx, target_ny = 60, 90  # about this many arrows across lon/lat
    lon_idx = np.linspace(0, len(lon_subset)-1, target_nx, dtype=int)
    lat_idx = np.linspace(0, len(lat_subset)-1, target_ny, dtype=int)
    
    lon_quiv = lon_subset[lon_idx]
    lat_quiv = lat_subset[lat_idx]
    u_quiv = u_full[np.ix_(lat_idx, lon_idx)]
    v_quiv = v_full[np.ix_(lat_idx, lon_idx)]

    # Time

    global_min = np.min(np.sqrt(u_full**2 + v_full**2))  # u_all, v_all should be full arrays (time, lat, lon)
    global_max = np.max(np.sqrt(u_full**2 + v_full**2))

    # Call plotting function with full fields for contours
    # and stepped ones for quivers
    plot_vector_frame(
        u_quiv, v_quiv, lat_quiv, lon_quiv,  # quiver data
        t, 
        quiver_scale=150,
        extent=extent
    )

In [ ]:
div = calc_divergence(hourly_composite)

# Loop for plotting divergence
for t in range(len(div['hour'])):
    div_time = div.isel(hour=t).values
    
    filestring = plot_divergence_frame(div_time, lat_subset, lon_subset, t,
                                 cluster=cluster,
                                 shapefile_gdf=gdf)
    
    print(f'Saved divergence plots: {filestring}')

In [ ]:
OW = okubo_weiss(hourly_composite, u_name="ua100m", v_name="va100m")['OW']

# Loop for plotting OW
for t in range(len(OW['hour'])):
    slice_time = OW.isel(hour=t).values
    
    filestring = plot_ow_frame(slice_time, lat_subset, lon_subset, t,
                                 cluster=cluster,
                                 shapefile_gdf=gdf)
    
    print(f'Saved OW plots: {filestring}')

In [ ]:
curl_z = calc_curl(hourly_composite)

for t in range(24):
    filestring = plot_curl_frame(curl_z[t,:,:],
                          lat_subset,
                          lon_subset,
                          t,
                          shapefile_gdf=gdf_clip)
    print(filestring)


In [ ]:
plot_windspeed_frame(max_speed, lat_subset, lon_subset,
                     field_type="max", cluster=cluster,
                     shapefile_gdf=gdf, extent=extent)

plot_windspeed_frame(min_speed, lat_subset, lon_subset,
                     field_type="min", cluster=cluster,
                     shapefile_gdf=gdf, extent=extent)

plot_windspeed_frame(diurnal_amp, lat_subset, lon_subset,
                     field_type="diff", cluster=cluster,
                     shapefile_gdf=gdf, extent=extent)
